# 7 · Comprehensions and Built-ins

*Built-in data structures* · 10 exercises

Write loops as expressions and let built-in functions do the work.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-7) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-7)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 7.1 · Filter inside a comprehension  `Core`

**Task.** Write `even_squares(nums)` returning the squares of the even numbers only, in one list comprehension.

**Example.** `even_squares([1, 2, 3, 4])` -> `[4, 16]`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-1)

In [ ]:
# 7.1 Filter inside a comprehension
def even_squares(nums):
    raise NotImplementedError

In [ ]:
def test_ex_7_1():
    assert even_squares([1, 2, 3, 4]) == [4, 16]


test_ex_7_1()
print('7.1 passed')

### 7.2 · Conditional expression inside a comprehension  `Core`

**Task.** Write `clamp_all(nums, lo, hi)` replacing values below lo with lo and above hi with hi, using `a if cond else b` inside the comprehension.

**Example.** `clamp_all([-5, 3, 12], 0, 10)` -> `[0, 3, 10]`

**Big O.** O(N).

**Note.** A trailing `if` filters items out; an `if ... else` before the `for` transforms every item.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-2)

In [ ]:
# 7.2 Conditional expression inside a comprehension
def clamp_all(nums, lo, hi):
    raise NotImplementedError

In [ ]:
def test_ex_7_2():
    assert clamp_all([-5, 3, 12], 0, 10) == [0, 3, 10]


test_ex_7_2()
print('7.2 passed')

### 7.3 · map and filter  `Core`

**Task.** Write `positive_ints(tokens)` that converts string tokens with `map(int, ...)` and keeps positives with `filter`.

**Example.** `positive_ints(["3", "-1", "0", "8"])` -> `[3, 8]`

**Big O.** O(N). Both return lazy iterators; wrap them in `list` to see the result.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-3)

In [ ]:
# 7.3 map and filter
def positive_ints(tokens):
    raise NotImplementedError

In [ ]:
def test_ex_7_3():
    assert positive_ints(['3', '-1', '0', '8']) == [3, 8]


test_ex_7_3()
print('7.3 passed')

### 7.4 · any and all  `Core`

**Task.** Write `all_positive(nums)` with `all`, `has_zero(nums)` with `any`, and `has_duplicate(nums)` using `len(set(...))`.

**Example.** `all_positive([1, 2])` -> `True`; `has_zero([3, 0])` -> `True`; `has_duplicate([1, 2, 1])` -> `True`

**Big O.** O(N); `any` and `all` stop early.

**Note.** `all([])` is True and `any([])` is False.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-4)

In [ ]:
# 7.4 any and all
def all_positive(nums):
    raise NotImplementedError


def has_zero(nums):
    raise NotImplementedError


def has_duplicate(nums):
    raise NotImplementedError

In [ ]:
def test_ex_7_4():
    assert all_positive([1, 2])
    assert has_zero([3, 0])
    assert has_duplicate([1, 2, 1])
    assert all_positive([])
    assert not has_zero([])
    assert not has_duplicate([1, 2])


test_ex_7_4()
print('7.4 passed')

### 7.5 · min and max with a key  `Core`

**Task.** Write `longest_word(words)` (first one wins on ties) and `closest_to_zero(nums)` (prefer the positive on ties), both with `key=`.

**Example.** `longest_word(["hi", "hello", "howdy"])` -> `"hello"`; `closest_to_zero([-4, 2, -2, 5])` -> `2`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-5)

In [ ]:
# 7.5 min and max with a key
def longest_word(words):
    raise NotImplementedError


def closest_to_zero(nums):
    raise NotImplementedError

In [ ]:
def test_ex_7_5():
    assert longest_word(['hi', 'hello', 'howdy']) == 'hello'
    assert closest_to_zero([-4, 2, -2, 5]) == 2
    assert closest_to_zero([-1, 3]) == -1


test_ex_7_5()
print('7.5 passed')

### 7.6 · sorted versus list.sort  `Core`

**Task.** Write `sorted_copy(arr)` that leaves `arr` untouched and `sort_in_place(arr)` that sorts it descending and returns `None`, like `list.sort`.

**Example.** `a = [3, 1, 2]; sorted_copy(a)` -> `[1, 2, 3]` and `a` is still `[3, 1, 2]`

**Big O.** O(N log N) each. `sorted` needs O(N) extra for the copy.

**Note.** `arr = arr.sort()` sets arr to None, a classic bug.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-6)

In [ ]:
# 7.6 sorted versus list.sort
def sorted_copy(arr):
    raise NotImplementedError


def sort_in_place(arr):
    raise NotImplementedError

In [ ]:
def test_ex_7_6():
    a = [3, 1, 2]
    assert sorted_copy(a) == [1, 2, 3]
    assert a == [3, 1, 2]
    assert sort_in_place(a) is None
    assert a == [3, 2, 1]


test_ex_7_6()
print('7.6 passed')

### 7.7 · Generator expressions save memory  `Core`

**Task.** Write `sum_of_squares(n)` for 1..n using a generator expression inside `sum(...)` (no list is built).

**Example.** `sum_of_squares(10)` -> `385`

**Big O.** O(N) time, O(1) memory. The list version needs O(N) memory.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-7)

In [ ]:
# 7.7 Generator expressions save memory
def sum_of_squares(n):
    raise NotImplementedError

In [ ]:
def test_ex_7_7():
    assert sum_of_squares(10) == 385


test_ex_7_7()
print('7.7 passed')

### 7.8 · Write a generator function  `Core`

**Task.** Write a generator `fib_gen()` that yields Fibonacci numbers forever, and `first_fibs(n)` that takes n of them.

**Example.** `first_fibs(8)` -> `[0, 1, 1, 2, 3, 5, 8, 13]`

**Big O.** O(1) per value.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-8)

In [ ]:
# 7.8 Write a generator function
def fib_gen():
    raise NotImplementedError


def first_fibs(n):
    raise NotImplementedError

In [ ]:
def test_ex_7_8():
    assert first_fibs(8) == [0, 1, 1, 2, 3, 5, 8, 13]


test_ex_7_8()
print('7.8 passed')

### 7.9 · zip for pairs of neighbours  `Core`

**Task.** Write `differences(arr)` returning `arr[i+1] - arr[i]` for each neighbouring pair, using `zip(arr, arr[1:])`.

**Example.** `differences([1, 4, 9, 16])` -> `[3, 5, 7]`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-9) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-9)

In [ ]:
# 7.9 zip for pairs of neighbours
def differences(arr):
    raise NotImplementedError

In [ ]:
def test_ex_7_9():
    assert differences([1, 4, 9, 16]) == [3, 5, 7]


test_ex_7_9()
print('7.9 passed')

### 7.10 · The walrus operator  `Practice`

**Task.** Write `read_until_blank(lines)` collecting lines (stripped) until the first empty one, using `:=` inside the `while` condition with an iterator.

**Example.** `read_until_blank(["a", "b", "", "c"])` -> `['a', 'b']`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-7-10) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-7-10)

In [ ]:
# 7.10 The walrus operator
def read_until_blank(lines):
    raise NotImplementedError

In [ ]:
def test_ex_7_10():
    assert read_until_blank(['a', 'b', '', 'c']) == ['a', 'b']
    assert read_until_blank(['x']) == ['x']


test_ex_7_10()
print('7.10 passed')